# 02 · Feature store

Calcula as features de **uma safra por vez** e grava em quatro tabelas, uma por bloco de comportamento. Cada safra é gravada com `replaceWhere`: rodar de novo uma data substitui só aquela data, e uma safra nova (ex.: out/26) entra sem recalcular as anteriores.

| Tabela | Blocos do MECE | Features |
| --- | --- | --- |
| `fs_transacoes` | A. Perfil · B. Volume · C. Tendência · E. Hábito · G. Loja | recência, frequência, intensidade, intervalos, horário, pontos |
| `fs_produtos` | D. Mix · E. Hábito (streak) | share de cada produto, produto dominante, streak |
| `fs_lives` | F. Comunidade | presença nas lives, lives perdidas, audiência |
| `fs_cursos` | H. Educação | cursos iniciados e finalizados, episódios, dias desde o último |

**Regra de todas as tabelas:** uma linha por usuário ativo nos 28 dias antes da `dtRef`, calculada só com dados **anteriores** à `dtRef`.

**Entrada:** `workspace.tmw_loyalty.*` e `workspace.tmw_education.*` · **Saída:** as quatro tabelas `fs_*` · **Próximo:** `02b_abt`

In [0]:
%run ./00_setup

## 1. Safras a calcular
Na primeira vez, todas as safras. Depois, só as novas: por exemplo, `SAFRAS_FS = ['2026-10-01']`.

In [0]:
SAFRAS_FS = [d.strftime('%Y-%m-%d') for d in pd.date_range(DT_INICIO, DT_SCORE, freq='MS')]
# SAFRAS_FS = ['2026-10-01']        # só uma safra nova

print(f'{len(SAFRAS_FS)} safras: {SAFRAS_FS[0]} a {SAFRAS_FS[-1]}')

## 2. Base comum
`tb_dia` tem todas as transações. `preparar_safra(dt)` cria a `tb_hist`: o histórico, até o dia anterior à `dtRef`, de quem esteve ativo nos 28 dias antes dela. Todas as tabelas partem daí.

In [0]:
spark.sql(f"""
    SELECT IdCliente, IdTransacao, DtCriacao, QtdePontos,
           to_date(DtCriacao)   AS dtDia,
           hour(DtCriacao)      AS hora,
           dayofweek(DtCriacao) AS diaSemana      -- 1 = domingo ... 7 = sábado
    FROM {T_TRANSACOES}
""").createOrReplaceTempView('tb_dia')


def preparar_safra(dt):
    """Cria a tb_hist da safra dt: histórico (antes de dt) de quem foi ativo nos 28 dias antes de dt."""
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW tb_hist AS
        WITH pop AS (
            SELECT DISTINCT IdCliente
            FROM tb_dia
            WHERE dtDia >= date_sub(DATE'{dt}', 28) AND dtDia < DATE'{dt}'
        )
        SELECT DATE'{dt}' AS dtRef, d.IdCliente, d.IdTransacao, d.DtCriacao, d.dtDia,
               d.hora, d.diaSemana, d.QtdePontos,
               datediff(DATE'{dt}', d.dtDia) AS diasAtras        -- 1 = ontem
        FROM pop p
        JOIN tb_dia d ON d.IdCliente = p.IdCliente AND d.dtDia < DATE'{dt}'
    """)


def populacao():
    """Usuários da safra atual (uma linha por usuário)."""
    return spark.sql('SELECT DISTINCT dtRef, IdCliente FROM tb_hist WHERE diasAtras <= 28').toPandas()


# colunas que nunca são nulas: ficam inteiras em todas as safras (o resto vira decimal)
INTEIROS = ['diasDesdePrimeira', 'recencia', 'diasAtivosD7', 'diasAtivosD14', 'diasAtivosD28', 'diasAtivosD56',
            'qtdTransacoesD28', 'pontosGanhosD28', 'freqVida', 'diasAtivosD28Anterior',
            'qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite', 'qtdDiasSemanaD56',
            'pontosGastosD28', 'saldoPontosD28', 'saldoPontos', 'qtdRetornos',
            'deltaDiasD28', 'flNovatoRisco', 'qtdProdutosDistintos', 'livesPerdidas']


def tipar(df):
    """Mesmo tipo em todas as safras: inteiro, decimal ou texto. Sem isso, o Delta recusa a gravação."""
    for c in df.columns:
        if c in ('dtRef', 'IdCliente'):
            continue
        if c in INTEIROS:
            df[c] = df[c].astype('int64')
        elif pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype('float64')
        else:
            df[c] = df[c].astype(str)
    return df


def gravar(df, tabela, dt):
    """Grava a safra dt na tabela, substituindo só essa data."""
    escrita = spark.createDataFrame(tipar(df)).write.format('delta').mode('overwrite')
    if spark.catalog.tableExists(tabela):
        escrita = escrita.option('replaceWhere', f"dtRef = DATE'{dt}'")
    escrita.saveAsTable(tabela)

## 3. Blocos
### 3.1 `fs_transacoes`: perfil, volume, tendência, hábito e loja

In [0]:
def fs_transacoes(dt):
    df = spark.sql(f"""
        WITH agg AS (
            SELECT dtRef, IdCliente,
                -- A. perfil
                MAX(diasAtras) AS diasDesdePrimeira,
                -- B. volume
                MIN(diasAtras) AS recencia,
                COUNT(DISTINCT CASE WHEN diasAtras <= 7  THEN dtDia END) AS diasAtivosD7,
                COUNT(DISTINCT CASE WHEN diasAtras <= 14 THEN dtDia END) AS diasAtivosD14,
                COUNT(DISTINCT CASE WHEN diasAtras <= 28 THEN dtDia END) AS diasAtivosD28,
                COUNT(DISTINCT CASE WHEN diasAtras <= 56 THEN dtDia END) AS diasAtivosD56,
                COUNT(CASE WHEN diasAtras <= 28 THEN IdTransacao END)    AS qtdTransacoesD28,
                SUM(CASE WHEN diasAtras <= 28 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS pontosGanhosD28,
                COUNT(DISTINCT dtDia) AS freqVida,
                -- C. tendência
                COUNT(DISTINCT CASE WHEN diasAtras BETWEEN 29 AND 56 THEN dtDia END) AS diasAtivosD28Anterior,
                -- E. hábito (horário e dia da semana)
                COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 0  AND 5  THEN 1 END) AS qtdMadrugada,
                COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 6  AND 11 THEN 1 END) AS qtdManha,
                COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 12 AND 17 THEN 1 END) AS qtdTarde,
                COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 18 AND 23 THEN 1 END) AS qtdNoite,
                COUNT(DISTINCT CASE WHEN diasAtras <= 56 THEN diaSemana END) AS qtdDiasSemanaD56,
                -- G. loja
                SUM(CASE WHEN diasAtras <= 28 AND QtdePontos < 0 THEN -QtdePontos ELSE 0 END) AS pontosGastosD28,
                SUM(CASE WHEN diasAtras <= 28 THEN QtdePontos ELSE 0 END) AS saldoPontosD28,
                SUM(QtdePontos) AS saldoPontos
            FROM tb_hist
            GROUP BY dtRef, IdCliente
        ),
        intervalos AS (          -- E. regularidade: dias entre um dia ativo e o seguinte (vida toda)
            SELECT IdCliente,
                   AVG(gap) AS avgIntervalo, STDDEV(gap) AS stdIntervalo, MAX(gap) AS maxIntervalo,
                   COUNT(CASE WHEN gap > 28 THEN 1 END) AS qtdRetornos
            FROM (
                SELECT IdCliente, datediff(dtDia, LAG(dtDia) OVER (PARTITION BY IdCliente ORDER BY dtDia)) AS gap
                FROM (SELECT DISTINCT IdCliente, dtDia FROM tb_hist)
            )
            GROUP BY IdCliente
        ),
        sessoes AS (             -- E. horas por dia ativo: da 1ª à última interação do dia (D28)
            SELECT IdCliente, AVG(duracao) AS horasPorDiaD28
            FROM (
                SELECT IdCliente, dtDia,
                       (unix_timestamp(MAX(DtCriacao)) - unix_timestamp(MIN(DtCriacao))) / 3600 AS duracao
                FROM tb_hist
                WHERE diasAtras <= 28
                GROUP BY IdCliente, dtDia
            )
            GROUP BY IdCliente
        )
        SELECT a.*, i.avgIntervalo, i.stdIntervalo, i.maxIntervalo, i.qtdRetornos, s.horasPorDiaD28,
               c.flEmail + c.flTwitch + c.flYouTube AS qtdRedes   -- A. redes conectadas (foto atual do cadastro)
        FROM agg a
        LEFT JOIN intervalos i ON a.IdCliente = i.IdCliente
        LEFT JOIN sessoes s    ON a.IdCliente = s.IdCliente
        LEFT JOIN {T_CLIENTES} c ON a.IdCliente = c.idCliente
        WHERE a.IdCliente IN (SELECT IdCliente FROM tb_hist WHERE diasAtras <= 28)
    """).toPandas()

    # C. tendência
    df['deltaDiasD28'] = df['diasAtivosD28'] - df['diasAtivosD28Anterior']
    df['tendencia'] = np.select([df['deltaDiasD28'] <= -2, df['deltaDiasD28'] >= 2], ['caindo', 'crescendo'], 'estável')
    df['razaoAtraso'] = df['recencia'] / df['avgIntervalo']                 # recência ÷ ritmo próprio
    # A. zona de risco dos novatos (3ª e 4ª semana de casa)
    df['flNovatoRisco'] = df['diasDesdePrimeira'].between(15, 28).astype(int)
    # B. intensidade
    df['transacoesPorDiaD28'] = df['qtdTransacoesD28'] / df['diasAtivosD28']
    # E. regularidade e período do dia
    df['cvIntervalo'] = df['stdIntervalo'] / df['avgIntervalo']
    periodos = ['qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite']
    df['periodoDominante'] = df[periodos].idxmax(axis=1).str.replace('qtd', '').str.lower()
    # B. posição relativa dentro da safra
    df['percentilPontos'] = df['pontosGanhosD28'].rank(pct=True)
    df['zScorePontos'] = (df['pontosGanhosD28'] - df['pontosGanhosD28'].mean()) / df['pontosGanhosD28'].std()
    # G. loja
    df['gastouD28'] = np.where(df['pontosGastosD28'] > 0, 'gastou', 'não gastou')
    return df

### 3.2 `fs_produtos`: mix de produtos e streak

In [0]:
# os 5 produtos que somam 98% dos itens; o resto vira 'outros' (ou 'desconhecido', fora do catálogo)
PRODUTOS_PRINCIPAIS = {
    'ChatMessage': 'chat',
    'Lista de presença': 'presenca',
    'Presença Streak': 'streak',
    'Resgatar Ponei': 'ponei',
    'Troca de Pontos StreamElements': 'troca',
}
GRUPOS = ['chat', 'desconhecido', 'outros', 'ponei', 'presenca', 'streak', 'troca']   # todas as safras com as mesmas colunas


def fs_produtos(dt):
    pop = populacao()
    produtos = spark.sql(f"""
        SELECT h.IdCliente,
               COALESCE(p.DescNomeProduto, 'desconhecido') AS produto,
               COUNT(DISTINCT h.IdTransacao) AS qtd                    -- distinct: transações com vários itens
        FROM tb_hist h
        LEFT JOIN {T_TRANSACAO_PRODUTO} tp ON h.IdTransacao = tp.IdTransacao
        LEFT JOIN {T_PRODUTOS} p           ON tp.IdProduto  = p.IdProduto
        WHERE h.diasAtras <= 28
        GROUP BY h.IdCliente, COALESCE(p.DescNomeProduto, 'desconhecido')
    """).toPandas()
    produtos['grupo'] = produtos['produto'].map(PRODUTOS_PRINCIPAIS).fillna(
        produtos['produto'].where(produtos['produto'] == 'desconhecido', 'outros'))

    pivot = (produtos.pivot_table(index='IdCliente', columns='grupo', values='qtd', aggfunc='sum', fill_value=0)
                     .reindex(columns=GRUPOS, fill_value=0))
    mix = pivot.div(pivot.sum(axis=1), axis=0).add_prefix('share_')
    mix['produtoDominante'] = pivot.idxmax(axis=1)
    mix['qtdProdutosDistintos'] = (pivot > 0).sum(axis=1)

    streak = spark.sql(f"""
        SELECT h.IdCliente, MIN(h.diasAtras) AS diasUltimoStreak
        FROM tb_hist h
        JOIN {T_TRANSACAO_PRODUTO} tp ON h.IdTransacao = tp.IdTransacao
        JOIN {T_PRODUTOS} p          ON tp.IdProduto  = p.IdProduto
        WHERE p.DescNomeProduto = 'Presença Streak' AND h.diasAtras <= 28
        GROUP BY h.IdCliente
    """).toPandas()

    df = pop.merge(mix.reset_index(), on='IdCliente', how='left').merge(streak, on='IdCliente', how='left')
    df['produtoDominante'] = df['produtoDominante'].fillna('sem produto')
    df['qtdProdutosDistintos'] = df['qtdProdutosDistintos'].fillna(0)
    df['fezStreak'] = np.where(df['diasUltimoStreak'].notna(), 'fez streak', 'não fez')
    return df

### 3.3 `fs_lives`: presença nas lives
Corte de **dia de live = 10 usuários**: no discovery, 25% dos dias têm até 7 usuários e a mediana é 64.

In [0]:
CORTE_LIVE = 10   # dias com >= 10 usuários = dia de live


def fs_lives(dt):
    return spark.sql(f"""
        WITH pop_dia AS (            -- usuários por dia, só antes da dtRef
            SELECT dtDia, COUNT(DISTINCT IdCliente) AS usuarios
            FROM tb_dia
            WHERE dtDia < DATE'{dt}'
            GROUP BY dtDia
        ),
        lives AS (                   -- nº de dias de live nos 28 dias antes da dtRef
            SELECT COUNT(*) AS qtdLivesD28
            FROM pop_dia
            WHERE dtDia >= date_sub(DATE'{dt}', 28) AND usuarios >= {CORTE_LIVE}
        ),
        dias_usuario AS (
            SELECT DISTINCT IdCliente, dtDia FROM tb_hist WHERE diasAtras <= 28
        ),
        presenca AS (
            SELECT d.IdCliente,
                   AVG(p.usuarios) AS audienciaMedia,
                   COUNT(CASE WHEN p.usuarios >= {CORTE_LIVE} THEN 1 END) / NULLIF(MAX(l.qtdLivesD28), 0) AS pctLivesPresente
            FROM dias_usuario d
            JOIN pop_dia p ON d.dtDia = p.dtDia
            CROSS JOIN lives l
            GROUP BY d.IdCliente
        ),
        perdidas AS (                -- recência medida em lives: dias de live desde a última presença
            SELECT u.IdCliente, COUNT(p.dtDia) AS livesPerdidas
            FROM (SELECT IdCliente, MAX(dtDia) AS ultimoDia FROM tb_hist GROUP BY IdCliente) u
            LEFT JOIN pop_dia p ON p.dtDia > u.ultimoDia AND p.usuarios >= {CORTE_LIVE}
            GROUP BY u.IdCliente
        )
        SELECT DATE'{dt}' AS dtRef, pr.IdCliente, pr.audienciaMedia, pr.pctLivesPresente, pe.livesPerdidas
        FROM presenca pr
        LEFT JOIN perdidas pe ON pr.IdCliente = pe.IdCliente
    """).toPandas()

### 3.4 `fs_cursos`: plataforma de cursos
A plataforma só tem histórico a partir de fev/2025. Antes de `DT_INICIO_CURSOS` as features ficam **nulas** (a plataforma não existia); depois, sem registro = não estudou (0).

In [0]:
def fs_cursos(dt):
    pop = populacao()
    contagens = ['qtdCursosIniciados', 'qtdCursosFinalizados', 'pctMedioCompleto', 'qtdEpsD28']

    if pd.Timestamp(dt) < pd.Timestamp(DT_INICIO_CURSOS):
        for c in contagens + ['diasDesdeUltimoEp', 'flFezCurso']:
            pop[c] = np.nan
        return pop

    cursos = spark.sql(f"""
        WITH eps AS (           -- 1ª conclusão de cada episódio
            SELECT u.idTMWCliente AS IdCliente, e.descSlugCurso, e.descSlugCursoEpisodio,
                   MIN(e.dtCriacao) AS dtCriacao
            FROM {T_EDU_USUARIOS} u
            JOIN {T_EDU_EPS_COMPLETOS} e ON u.idUsuario = e.idUsuario
            GROUP BY u.idTMWCliente, e.descSlugCurso, e.descSlugCursoEpisodio
        ),
        total_eps AS (          -- nº de episódios de cada curso no catálogo
            SELECT descSlugCurso, COUNT(*) AS qtdEpsCurso
            FROM {T_EDU_EPS}
            GROUP BY descSlugCurso
        ),
        por_curso AS (          -- avanço em cada curso, só com o passado da dtRef
            SELECT IdCliente, descSlugCurso,
                   COUNT(*)                                                       AS qtdEps,
                   COUNT(CASE WHEN dtCriacao >= date_sub(DATE'{dt}', 28) THEN 1 END) AS qtdEpsD28,
                   MAX(dtCriacao)                                                 AS dtUltimoEp
            FROM eps
            WHERE dtCriacao < DATE'{dt}'
            GROUP BY IdCliente, descSlugCurso
        )
        SELECT p.IdCliente,
               COUNT(*)                                              AS qtdCursosIniciados,
               COUNT(CASE WHEN p.qtdEps >= t.qtdEpsCurso THEN 1 END) AS qtdCursosFinalizados,
               AVG(LEAST(p.qtdEps / t.qtdEpsCurso, 1))               AS pctMedioCompleto,
               SUM(p.qtdEpsD28)                                      AS qtdEpsD28,
               datediff(DATE'{dt}', MAX(p.dtUltimoEp))               AS diasDesdeUltimoEp
        FROM por_curso p
        LEFT JOIN total_eps t ON p.descSlugCurso = t.descSlugCurso
        GROUP BY p.IdCliente
    """).toPandas()

    df = pop.merge(cursos, on='IdCliente', how='left')
    df[contagens] = df[contagens].fillna(0)                 # sem registro = não estudou
    df['flFezCurso'] = (df['qtdCursosIniciados'] > 0).astype(float)
    return df                                                

## 4. Gravar
Para cada safra: monta a `tb_hist`, calcula os quatro blocos e grava cada um na sua tabela.

In [0]:
BLOCOS_FS = {
    FS_TRANSACOES: fs_transacoes,
    FS_PRODUTOS:   fs_produtos,
    FS_LIVES:      fs_lives,
    FS_CURSOS:     fs_cursos,
}

for dt in SAFRAS_FS:
    preparar_safra(dt)
    linhas = {}
    for tabela, calcular in BLOCOS_FS.items():
        df = calcular(dt)
        gravar(df, tabela, dt)
        linhas[tabela.split('.')[-1]] = len(df)
    print(dt, linhas)

## 5. Conferência
Cada tabela deve ter as mesmas safras e o mesmo número de usuários por safra.

In [0]:
conferencia = None
for tabela in BLOCOS_FS:
    n = spark.table(tabela).groupBy('dtRef').count().withColumnRenamed('count', tabela.split('.')[-1]).toPandas()
    conferencia = n if conferencia is None else conferencia.merge(n, on='dtRef', how='outer')
conferencia = conferencia.sort_values('dtRef').set_index('dtRef')
conferencia['iguais'] = conferencia.nunique(axis=1) == 1
assert conferencia['iguais'].all(), 'Alguma safra tem número de usuários diferente entre as tabelas'
conferencia